# Load Data


In [1]:
import pandas as pd
import glob
import numpy as np

files = glob.glob("../data/raw/202606-citibike-tripdata/*.csv")
df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

print(files)

/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_68174/3073734667.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_68174/3073734667.py:6: DtypeWarning: Columns (5,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_68174/3073734667.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
/var/folders/dr/r55rrk2n0jsfmys_4px8ctxm0000gn/T/ipykernel_68174/3073734667.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in files), ignore_index=

['../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_6.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_5.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_4.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_1.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_3.csv', '../data/raw/202606-citibike-tripdata/202606-citibike-tripdata_2.csv']


In [2]:
# Display basic information about the DataFrame
print(df.columns)
print(df.shape)

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='object')
(5384468, 13)


In [3]:
# Display basic information about the DataFrame
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5384468 entries, 0 to 5384467
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 534.0+ MB
None


In [4]:
print(df.head())

            ride_id  rideable_type               started_at  \
0  E8950F16E8963131  electric_bike  2026-06-18 08:54:16.609   
1  AB5E676A98A98FFD  electric_bike  2026-06-18 18:10:49.259   
2  7ACF812D3851E83B  electric_bike  2026-06-30 18:11:01.624   
3  66955C7329D73810  electric_bike  2026-06-22 07:37:19.887   
4  1020978B6F86D90F  electric_bike  2026-06-30 07:45:23.969   

                  ended_at      start_station_name start_station_id  \
0  2026-06-18 09:06:26.757          E 6 St & Ave D          5506.14   
1  2026-06-18 18:13:07.198  Broadway & Roebling St          5125.07   
2  2026-06-30 18:27:34.826  Columbus Ave & W 59 St          6986.07   
3  2026-06-22 07:52:03.386         E 14 St & Ave B          5736.09   
4  2026-06-30 07:59:28.521         E 14 St & Ave B          5736.09   

            end_station_name end_station_id  start_lat  start_lng    end_lat  \
0            E 51 St & 2 Ave        6575.03  40.722281 -73.976687  40.755293   
1        Broadway & Berry St      

In [5]:
# Check for missing values in the DataFrame
print(df.isna().sum())

ride_id                   0
rideable_type             0
started_at                0
ended_at                  0
start_station_name     3154
start_station_id       3154
end_station_name      13454
end_station_id        14211
start_lat              3154
start_lng              3154
end_lat               14190
end_lng               14190
member_casual             0
dtype: int64


In [6]:
print(f"Duplicate ride_ids: {df['ride_id'].duplicated().sum()}")

Duplicate ride_ids: 0


In [7]:
# Display value counts for 'member_casual' and 'rideable_type' columns, no unexpected categories found
print(df['member_casual'].value_counts())
print(df['rideable_type'].value_counts())

member_casual
member    4278486
casual    1105982
Name: count, dtype: int64
rideable_type
electric_bike    3883419
classic_bike     1501049
Name: count, dtype: int64


# Handle Missing Data

In [8]:
print(f"Before dropping: {df.shape[0]} rows")

Before dropping: 5384468 rows


In [9]:
# Confirm that missingness in start station name and coordinates is aligned (i.e., if one is missing, the other is also missing)
same_rows = (df['start_station_name'].isna() == df['start_lat'].isna()).all()
print(f"Start station name/coord missingness aligned: {same_rows}")

Start station name/coord missingness aligned: True


In [10]:
# Drop rows with missing start coordinates (can't map these trips)
df = df[df['start_lat'].notna() & df['start_lng'].notna()]
print(f"After dropping missing start coords: {df.shape[0]} rows")

# Drop rows with missing end coordinates (can't map these trips)
df = df[df['end_lat'].notna() & df['end_lng'].notna()]
print(f"After dropping missing end coords: {df.shape[0]} rows")

# member_casual has 0 missing values in this dataset, but kept as a safeguard
# since it's the core variable for the whole analysis
df = df[df['member_casual'].notna()]
print(f"After dropping missing member_casual: {df.shape[0]} rows")

After dropping missing start coords: 5381314 rows
After dropping missing end coords: 5367618 rows
After dropping missing member_casual: 5367618 rows


In [11]:
# Rows that have end coordinates but no end_station_id/name (dockless e-bike returns)
# Flag them instead of dropping, since we still have usable lat/lng
no_end_station = df['end_station_id'].isna() & df['end_lat'].notna()
df['end_is_dockless'] = no_end_station
print(f"Rows kept with coordinates but no end station (dockless returns): {no_end_station.sum()}")

print(f"\nFinal shape after Step 3: {df.shape}")

Rows kept with coordinates but no end station (dockless returns): 21

Final shape after Step 3: (5367618, 14)


In [12]:
# Rows that have start coordinates but no start_station_id/name (dockless e-bike starts)
no_start_station = df['start_station_id'].isna() & df['start_lat'].notna()
df['start_is_dockless'] = no_start_station
print(f"Rows with start coordinates but no start station: {no_start_station.sum()}")

Rows with start coordinates but no start station: 0


# Parse Timestamps

In [13]:
# Convert start and end timestamps to datetime objects
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])

print(df[['started_at', 'ended_at']].head())


               started_at                ended_at
0 2026-06-18 08:54:16.609 2026-06-18 09:06:26.757
1 2026-06-18 18:10:49.259 2026-06-18 18:13:07.198
2 2026-06-30 18:11:01.624 2026-06-30 18:27:34.826
3 2026-06-22 07:37:19.887 2026-06-22 07:52:03.386
4 2026-06-30 07:45:23.969 2026-06-30 07:59:28.521


In [14]:
df['start_hour'] = df['started_at'].dt.hour
df['start_dow'] = df['started_at'].dt.dayofweek  # 0=Monday
df['is_weekend'] = df['start_dow'].isin([5, 6])

# Compute Trip Duration

In [15]:
# Compute ended_at - started_at to get trip duration
df['trip_duration'] = (df['ended_at'] - df['started_at']).dt.total_seconds() 
print(df['trip_duration'].describe().round(3),"\n\n") # Show times in seconds
print((df['trip_duration']/60).describe().round(3)) # Show times in minutes


count    5367618.000
mean         809.650
std         1138.022
min           60.012
25%          341.042
50%          585.677
75%         1002.746
max        89997.767
Name: trip_duration, dtype: float64 


count    5367618.000
mean          13.494
std           18.967
min            1.000
25%            5.684
50%            9.761
75%           16.712
max         1499.963
Name: trip_duration, dtype: float64


In [16]:
df['is_long_trip'] = df['trip_duration'] > 14400  # over 4 hours, adjust threshold as you see fit
print(f"Trips over 4 hours: {df['is_long_trip'].sum()}")

Trips over 4 hours: 2627


# Compute Trip Distance

In [17]:
# Haversine formula to compute distance between two lat/lng points
def haversine(lat1, lon1, lat2, lon2):
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a)) 
    r = 6371  # Radius of earth in kilometers. Use 3956 for miles
    return c * r

haversine_vectorized = np.vectorize(haversine)
print("Computing trip distances using Haversine formula...")
df['trip_distance_km'] = haversine_vectorized(df['start_lat'], df['start_lng'], df['end_lat'], df['end_lng'])
print(df['trip_distance_km'].describe().round(3),"\n\n") # Show distances in kilometers


Computing trip distances using Haversine formula...
count    5367618.000
mean           2.114
std            1.783
min            0.000
25%            0.903
50%            1.594
75%            2.779
max           28.183
Name: trip_distance_km, dtype: float64 




In [18]:
# Print count of trips with zero distance (start and end coordinates are the same)
zero_distance_count = (df['trip_distance_km'] == 0).sum()
print(f"Number of trips with zero distance: {zero_distance_count}")

# May be worth investigating these trips further.
# Could be round trips, data entry errors, or other anomalies. 
# For now, we will keep them in the dataset.

Number of trips with zero distance: 121729


# Filter Out Obviously Bad Rows
eg. very short trips (under ~1 minute might be false starts or not real rides)

In [19]:
# If trip duration is under 60 seconds, flag it as a potential false start or error
df['is_short_trip'] = df['trip_duration'] < 60
print(f"Number of trips flagged as short trips (under 60 seconds): {df['is_short_trip'].sum()}")

# Confirms City Bike dataset likely already removed any trips less than one minute

Number of trips flagged as short trips (under 60 seconds): 0


# Investigating station ID formatting inconsistencies

In [20]:
# Same station name mapping to multiple station IDs — checking if this is a
# data error or genuinely different stations
station_name_counts = df.groupby('start_station_name')['start_station_id'].nunique()
duplicate_station_names = station_name_counts[station_name_counts > 1].index.tolist()
print(f"Station names with multiple IDs: {len(duplicate_station_names)}")

for station_name in duplicate_station_names[:5]:  # sample check
    station_ids = df[df['start_station_name'] == station_name]['start_station_id'].unique()
    print(f"Station Name: {station_name}, Station IDs: {station_ids}")

Station names with multiple IDs: 1324
Station Name: 1 Ave & E 110 St, Station IDs: ['7522.02' 7522.02]
Station Name: 1 Ave & E 118 St, Station IDs: ['7596.11' 7596.11]
Station Name: 1 Ave & E 16 St, Station IDs: ['5779.08' 5779.08]
Station Name: 1 Ave & E 18 St, Station IDs: ['5854.09' 5854.09]
Station Name: 1 Ave & E 30 St, Station IDs: ['6079.03' 6079.03]


Same station name, IDs differing only in decimal precision (e.g. `7386.10`
vs `7386.1`) — same coordinates confirmed, so this is a string-formatting
inconsistency, not two physical docks.

Also found: trailing underscores on some IDs (`6517.08_`), an `_OLD` suffix
on others (`3184.07_OLD`), and non-numeric system/depot codes (`SYS038`,
`HB101`, `JC076`, `Shop Morgan`). Investigating each below.

In [21]:
# Trailing underscore and _OLD suffix — check if _OLD represents the same
# physical station re-issued a new ID, or an actual relocation
old_suffix_ids = df[df['start_station_id'].str.contains('_OLD', na=False)]['start_station_id'].unique()
for sid in old_suffix_ids:
    base_id = sid.replace('_OLD', '')
    old_coords = df[df['start_station_id'] == sid][['start_lat', 'start_lng']].drop_duplicates()
    current_coords = df[df['start_station_id'] == base_id][['start_lat', 'start_lng']].drop_duplicates()
    print(f"{sid} vs {base_id}")
    print("OLD:", old_coords.values, " Current:", current_coords.values)

3184.07_OLD vs 3184.07
OLD: [[ 40.64624929 -73.99452601]]  Current: [[ 40.64624929 -73.99452601]]


Coordinates match exactly for every `_OLD` case — same physical station,
re-issued a new system ID. Safe to merge by stripping the suffix.

In [22]:
# Apply the confirmed-safe cleanup: strip trailing underscore, merge _OLD suffix
for col in ['start_station_id', 'end_station_id']:
    df[col] = (
        df[col]
        .astype(str)
        .str.rstrip('_')
        .str.replace('_OLD', '', regex=False)
    )

In [23]:
# Flag remaining non-standard IDs (system/depot codes, other cities) rather
# than forcing a numeric conversion — station_id doesn't need to be numeric
# for grouping or mapping to work
standard_pattern = r'^\d+\.?\d*$'
df['start_id_nonstandard'] = ~df['start_station_id'].str.match(standard_pattern, na=False)
df['end_id_nonstandard'] = ~df['end_station_id'].str.match(standard_pattern, na=False)

print(f"Non-standard start IDs: {df['start_id_nonstandard'].sum()}")
print(f"Non-standard end IDs: {df['end_id_nonstandard'].sum()}")

print(df[df['start_id_nonstandard']]['start_station_id'].unique())
print(df[df['end_id_nonstandard']]['end_station_id'].unique())

Non-standard start IDs: 195
Non-standard end IDs: 686
['SYS038' 'Shop Morgan ' 'SYS016' 'SYS033']
['HB203' 'Shop Morgan ' 'SYS038' 'HB101' 'HB611' 'HB603' 'HB502' 'JC076'
 'HB608' 'JC102' 'JC149' 'HB201' 'HB409' 'JC116' 'JC057' 'HB105' 'JC009'
 'SYS016' 'JC013' 'HB103' 'JC014' 'JC003' 'HB602' 'nan' 'JC072' 'HB505'
 'JC002' 'JC038' 'HB503' 'JC035' 'JC110' 'JC117' 'JC105' 'JC006' 'JC008'
 'JC059' 'HB501' 'JC032' 'HB305' 'JC075' 'JC065' 'JC097' 'HB202' 'JC139'
 'HB610' 'HB609' 'HB401' 'HB106' 'SYS033' 'HB612' 'JC098' 'HB304' 'JC145'
 'JC115' 'JC099' 'JC109' 'JC027' 'HB408' 'HB303' 'JC095' 'JC066' 'JC052']


In [24]:
# Confirm each non-standard ID still maps to exactly one consistent lat/lng —
# if so, these rows are mappable and usable even without a "clean" numeric ID
for col, lat, lng, flag in [
    ('start_station_id', 'start_lat', 'start_lng', 'start_id_nonstandard'),
    ('end_station_id', 'end_lat', 'end_lng', 'end_id_nonstandard'),
]:
    check = df[df[flag]].groupby(col)[[lat, lng]].nunique()
    inconsistent = check[(check[lat] > 1) | (check[lng] > 1)]
    print(f"{col}: non-standard IDs with inconsistent coordinates: {len(inconsistent)}")

start_station_id: non-standard IDs with inconsistent coordinates: 0
end_station_id: non-standard IDs with inconsistent coordinates: 1


In [25]:
# Apply the confirmed-safe cleanup: strip trailing underscore, merge _OLD suffix
for col in ['start_station_id', 'end_station_id']:
    df[col] = (
        df[col]
        .astype(str)
        .str.rstrip('_')
        .str.replace('_OLD', '', regex=False)
        .replace('nan', pd.NA)   # undo the str(NaN) -> "nan" conversion
    )

# Confirm each non-standard ID still maps to exactly one consistent lat/lng —
# if so, these rows are mappable and usable even without a "clean" numeric ID
check_end = df[df['end_id_nonstandard']].groupby('end_station_id')[['end_lat', 'end_lng']].nunique()
inconsistent_end = check_end[(check_end['end_lat'] > 1) | (check_end['end_lng'] > 1)]
print(inconsistent_end)


Empty DataFrame
Columns: [end_lat, end_lng]
Index: []


In [26]:
# Exclude Hoboken/Jersey City and depot/system codes to keep scope to just NYC
out_of_scope_pattern = r'^(HB|JC|SYS|Shop)'
out_of_scope_mask = (
    df['start_station_id'].str.match(out_of_scope_pattern, na=False) |
    df['end_station_id'].str.match(out_of_scope_pattern, na=False)
)
print(f"Trips excluded (Hoboken/Jersey City/depot): {out_of_scope_mask.sum()} ({out_of_scope_mask.mean()*100:.3f}%)")

df = df[~out_of_scope_mask]
print(f"Shape after excluding out-of-scope stations: {df.shape}")

# Convert to float - resolves decimal-precision duplicates
# (e.g. '7386.10' vs '7386.1') and confirm nothing non-numeric remains
df['start_station_id'] = pd.to_numeric(df['start_station_id'], errors='coerce')
df['end_station_id'] = pd.to_numeric(df['end_station_id'], errors='coerce')

# compare against how many were pd.NA before conversion (the dockless-return count)
original_nan = df['end_station_id'].isna().sum()
print(f"end_station_id NaN before conversion: {original_nan}")
newly_nan = df['end_station_id'].isna().sum()
print(f"end_station_id NaN after conversion: {newly_nan}")

print(df['start_station_id'].apply(type).value_counts())
print(df['end_station_id'].apply(type).value_counts())

Trips excluded (Hoboken/Jersey City/depot): 827 (0.015%)
Shape after excluding out-of-scope stations: (5366791, 24)
end_station_id NaN before conversion: 21
end_station_id NaN after conversion: 21
start_station_id
<class 'float'>    5366791
Name: count, dtype: int64
end_station_id
<class 'float'>    5366791
Name: count, dtype: int64


# Save Cleaned Output

In [27]:
# save the cleaned dataframe as parquet for efficient storage and retrieval
output_path = "../data/processed/cleaned_citibike_data.parquet"

for col in ['start_station_name', 'end_station_name']:
    df[col] = df[col].astype('string')

df.to_parquet(output_path, index=False)
print(f"Cleaned data saved to {output_path}")

Cleaned data saved to ../data/processed/cleaned_citibike_data.parquet
